In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [3]:
#____________________________ part a ______________________________#
import os
import zipfile
import glob
import cv2 as cv
import numpy as np

# Setting the common size of the images
SIZE = (224, 224)

images = []
labels = []

target_folder = './fire_dataset'

if not os.path.exists(target_folder):
    with zipfile.ZipFile('fire_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall(target_folder)
        print('Zip file extracted successfully.')
else:
    print('Target folder already exists, skipping extraction.')

# Looping over the directories to get the file paths and load the images
for label, folder in enumerate(['fire_images', 'non_fire_images']):
    for file_path in glob.glob(f'fire_dataset/{folder}/*.png'):
        image = cv.imread(file_path)
        image = cv.resize(image, SIZE)
        image = cv.normalize(image, None, alpha=0, beta=1, norm_type=cv.NORM_MINMAX, dtype=cv.CV_32F)
        images.append(image)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print('Shape of images:', images.shape)
print('Shape of labels:', labels.shape)


Target folder already exists, skipping extraction.


Corrupt JPEG data: 1 extraneous bytes before marker 0xd9


Shape of images: (998, 224, 224, 3)
Shape of labels: (998,)


In [4]:
#____________________________ part b ______________________________#
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=15)

# the maximum dimension of X_train and X_test should be 2 so as to it can be given to a logistic regression model 
X_train = X_train.reshape(len(X_train), -1)
X_test = X_test.reshape(len(X_test), -1)

print('Shape of X_train:', X_train.shape)
print('Shape of y_train:', y_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)


Shape of X_train: (798, 150528)
Shape of y_train: (798,)
Shape of X_test: (200, 150528)
Shape of y_test: (200,)


In [5]:
#____________________________ part c ______________________________#
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=800,random_state=15)
model.fit(X_train, y_train)


LogisticRegression(max_iter=800, random_state=15)

In [6]:
#____________________________ part d ______________________________#
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
confusion_matrix = confusion_matrix(y_test, y_pred)
print('Accuracy:', accuracy)
print('Confusion Matrix:')
print(confusion_matrix)


Accuracy: 0.93
Confusion Matrix:
[[145   4]
 [ 10  41]]


In [7]:
#____________________________ part e ______________________________#
from sklearn.metrics import confusion_matrix

# Getting the predicted probabilities for the positive class (fire) on the training data
y_train_proba = model.predict_proba(X_train)[:, 1]

# Evaluating the model's performance at different probability thresholds
thresholds = np.linspace(0, 1, 101)
accuracies = []
for threshold in thresholds:
    y_train_pred = (y_train_proba >= threshold).astype(int)
    accuracy = np.mean(y_train_pred == y_train)
    accuracies.append(accuracy)

# Selecting the probability threshold that gives the highest accuracy on the training data
best_threshold = thresholds[np.argmax(accuracies)]

# Getting the predicted probabilities for the positive class (fire) on the test data
y_test_proba = model.predict_proba(X_test)[:, 1]

# Reporting the accuracy and confusion matrix on the test data using the best threshold for training data
y_test_pred = (y_test_proba >= best_threshold).astype(int)
accuracy = np.mean(y_test_pred == y_test)
cm = confusion_matrix(y_test, y_test_pred)
print('Accuracy on test data:', accuracy)
print('Confusion matrix on test data:')
print(cm)


Accuracy on test data: 0.905
Confusion matrix on test data:
[[134  15]
 [  4  47]]


In [8]:
#____________________________ part f ______________________________#
import pickle

# Saving the trained model
with open('log_reg_1.pkl', 'wb') as f:
    pickle.dump(model, f)

%run classifier.py
